<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 6 · Black–Litterman and Bayesian Portfolio Construction

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
We implement the Black–Litterman workflow:
- compute equilibrium (implied) returns from a benchmark,
- encode absolute and relative views,
- blend priors and views into posterior returns, and
- compare resulting portfolios/diagnostics.


### Getting Help While Using Black–Litterman
- **Chapter 4** for mean–variance background.
- **Appendix A** for the derivation of the update formulas.
- **Appendix B** to refresh NumPy array manipulations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"
RISK_AVERSION = 3.0
TAU = 0.05

## 1. Inputs: Returns, Covariance, Benchmark Weights
We build annualized statistics for six assets and assume equal benchmark weights.

In [ ]:
assets = ["AAPL", "NVDA", "JPM", "SPY", "GLD", "TLT"]
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
prices = prices.ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()[assets]
mean_returns = log_rets.mean() * 252
cov_matrix = log_rets.cov() * 252
w_benchmark = np.repeat(1 / len(assets), len(assets))
mean_returns

## 2. Equilibrium (Implied) Returns
Black–Litterman starts by inferring returns that make the benchmark optimal under mean–variance preferences.

In [ ]:
pi = RISK_AVERSION * cov_matrix.values @ w_benchmark
pd.Series(pi, index=assets)

## 3. Encode Investor Views
We mix one absolute view (AAPL +1&#37; vs. cash) and one relative view (NVDA outperform GLD by 50 bps).

In [ ]:
P = np.array(
    [
        [1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, -1, 0],
    ]
)
Q = np.array([0.01, 0.005])
OMEGA = np.diag([0.0004, 0.0009])

## 4. Black–Litterman Update Function
The function below returns posterior means and covariances.

In [ ]:
def black_litterman(cov: np.ndarray, tau: float, pi_vec: np.ndarray, P: np.ndarray,
Q: np.ndarray, omega: np.ndarray):
    tau_cov = tau * cov
    inv_tau_cov = np.linalg.inv(tau_cov)
    middle = P.T @ np.linalg.inv(omega) @ P
    posterior_cov = np.linalg.inv(inv_tau_cov + middle)
    posterior_mean = posterior_cov @ (inv_tau_cov @ pi_vec + P.T @ np.linalg.inv(omega) @
        Q)
    return posterior_mean, posterior_cov

post_mean, post_cov = black_litterman(
    cov_matrix.values,
    TAU,
    pi,
    P,
    Q,
    OMEGA,
)
pd.Series(post_mean, index=assets)

## 5. Compare Prior vs. Posterior Inputs
We chart implied vs. posterior returns to see how views shift expectations.

In [ ]:
comparison = pd.DataFrame({"pi": pi, "posterior": post_mean}, index=assets)
comparison

## 6. Posterior Portfolio Illustration
We compute max-Sharpe weights based on posterior means and compare with benchmark weights.

In [ ]:
cov_inv = np.linalg.inv(post_cov)
ones = np.ones(len(assets))
posterior_weights = cov_inv @ (post_mean - 0.02 * ones)
posterior_weights /= posterior_weights.sum()
weights_df = pd.DataFrame(
    {
        "Benchmark": w_benchmark,
        "Posterior": posterior_weights,
    },
    index=assets,
)
weights_df

In [ ]:
ax = weights_df.plot(kind="bar", figsize=(12, 6))
ax.set_ylabel("Weight")
ax.set_title("Benchmark vs. Black–Litterman Posterior Weights")
plt.show()

## 7. Posterior Frontier Snapshot
Monte Carlo sampling conveys how the posterior pushes the efficient frontier.

We overlay Monte Carlo samples based on the equilibrium prior (pi) and the posterior to visualize how views reshape the feasible set.

In [ ]:
rng = np.random.default_rng(123)
n_ports = 4000
weights = rng.dirichlet(np.ones(len(assets)), size=n_ports)
prior_returns = weights @ pi
prior_vols = np.sqrt(np.einsum("bi,ij,bj->b", weights, cov_matrix.values, weights))
post_returns = weights @ post_mean
post_vols = np.sqrt(np.einsum("bi,ij,bj->b", weights, post_cov, weights))
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(
    prior_vols,
    prior_returns,
    s=6,
    alpha=0.6,
    color=plt.cm.coolwarm(0.2),
    label="Prior",
)
ax.scatter(
    post_vols,
    post_returns,
    s=6,
    alpha=0.7,
    color=plt.cm.coolwarm(0.8),
    label="Posterior",
)
ax.set_xlabel("Volatility")
ax.set_ylabel("Expected return")
ax.set_title("Prior vs. Posterior Monte Carlo Frontiers")
ax.legend()
plt.show()

## 8. Exercises
### Exercise 1 – View Confidence Sweep
Vary the diagonal of <code>OMEGA</code> to see how posterior means react.
<details><summary>Hint</summary>
Multiply <code>OMEGA</code> by scalars (0.25, 1, 4) and recompute <code>posterior_mean</code>.
</details>

### Exercise 2 – Alternative Benchmark
Swap <code>w_benchmark</code> for market-cap weights proportional to the last price and repeat the workflow.
<details><summary>Hint</summary>
Use <code>prices.iloc[-1]</code> as a proxy for cap weight, normalize, and rebuild <code>pi</code>.
</details>

### Exercise 3 – Frontier Comparison
Plot Monte Carlo clouds for both prior (<code>pi</code>) and posterior means on the same axes.
<details><summary>Hint</summary>
Reuse the Chapter 4 sampling function and overlay scatter plots with different colors.
</details>


## 9. Takeaways for Chapter 6
- Black–Litterman tempers mean estimates before optimization.
- Encoding views as matrices keeps the workflow auditable.
- Posterior outputs feed directly into the Chapter 4 optimization toolkit.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">